Formatação inicial de Dataframe Unindo os DataSets 

In [68]:
import pandas as pd
import glob
import os

caminho_pasta = 'dados_inmet'
arquivos = glob.glob(os.path.join(caminho_pasta, "*.CSV"))

lista_final = []

print(f"--- Iniciando Unificação de {len(arquivos)} arquivos ---\n")

for f in arquivos:
    try:
        # 1. Lendo o arquivo bruto
        df_temp = pd.read_csv(f, sep=';', encoding='latin-1', skiprows=8, decimal=',', index_col=False)
        df_temp.columns = df_temp.columns.str.strip().str.upper()
        
        # 2. Localização Dinâmica de Colunas (O segredo para não ter coluna vazia)
        # Procuramos colunas que CONTENHAM os termos, independente do nome completo
        def encontrar_coluna(termos):
            for termo in termos:
                for col in df_temp.columns:
                    if termo in col: return col
            return None

        # Mapeamento
        mapa = {
            'DATA': encontrar_coluna(['DATA']),
            'HORA': encontrar_coluna(['HORA']),
            'CHUVA': encontrar_coluna(['PRECIPITAÇÃO', 'CHUVA']),
            'PRESSAO': encontrar_coluna(['PRESSAO ATMOSFERICA']),
            'TEMP': encontrar_coluna(['TEMPERATURA DO AR - BULBO SECO', 'TEMP_AR']),
            'UMID': encontrar_coluna(['UMIDADE RELATIVA']),
            'VENTO': encontrar_coluna(['RAJADA', 'VENTO_RAJADA']),
            'RAD': encontrar_coluna(['RADIACAO'])
        }

        # 3. Filtragem e Renomeação
        # Removemos os que não foram encontrados (None)
        mapa_limpo = {v: k for k, v in mapa.items() if v is not None}
        df_filtrado = df_temp[list(mapa_limpo.keys())].rename(columns=mapa_limpo)
        
        # 4. Limpeza de Linhas Inúteis
        # Removemos linhas onde a data é nula ou a hora está mal formada
        df_filtrado = df_filtrado.dropna(subset=['DATA', 'HORA'])
        
        #limpeza de linhas vazias 
        limiar_minimo = 4

        #Definimos que a linha precisa de pelo menos 4 valores REAIS para ser mantida
        # (Ex: Data, Hora, Temperatura e Humidade). Se tiver menos que isso, é lixo.
        df_filtrado = df_filtrado.dropna(thresh=limiar_minimo)
        
        #Especificamente para 2025, removemos linhas onde os sensores principais estão vazios
        df_filtrado = df_filtrado.dropna(subset=['CHUVA', 'TEMP'], how='all')

        lista_final.append(df_filtrado)
        print(f"✅ {os.path.basename(f)}: {len(df_filtrado)} linhas e {df_filtrado.columns.tolist()} colunas.")

    except Exception as e:
        print(f"❌ Erro no arquivo {f}: {e}")        

# Juntando tudo no Dataset Mestre
if lista_final:
    df_master = pd.concat(lista_final, axis=0, ignore_index=True)
    
    # Ordenar por data para não ficar bagunçado
    df_master = df_master.sort_values(['DATA', 'HORA'])
    
else:
    print("\nERRO: Nenhum arquivo foi lido corretamente.")

--- Iniciando Unificação de 10 arquivos ---

✅ INMET_NE_CE_A305_FORTALEZA_01-01-2017_A_31-12-2017.CSV: 8760 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2018_A_31-12-2018.CSV: 8760 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2019_A_31-12-2019.CSV: 6356 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2020_A_31-12-2020.CSV: 8777 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2021_A_31-12-2021.CSV: 8179 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2022_A_31-12-2022.CSV: 8712 linhas e ['DATA', 'HORA', 'CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD'] colunas.
✅ INMET_NE_CE_A305_FORTALEZA_01-01-2023_A_31-12-2023.CS

Conversão para horário local e formatação de data e hora

In [69]:
#convertendo string para int
df_master['HORA_NUM'] = df_master['HORA'].str.extract(r'(\d+)').astype(int)

#A formatação das horas de 2019 a 2025 são diferentes dos outros anos,
#precisamos fazer a padronização
df_master['HORA_NUM'] = df_master['HORA_NUM'].apply(lambda x: x // 100 if x >= 100 else x)

#Agora precisamos padronizar a coluna 'HORA' para o formato Brasileiro horário local

#retirando do formato UTC e unindo as colunas 'DATA' e 'HORA'
df_master['dt_utc'] = pd.to_datetime(
    df_master['DATA'].str.replace('/', '-') + ' ' +
    df_master['HORA_NUM'].astype(str).str.zfill(2) + ':00'
                                )
#Convertendo para o fuso horário de fortaleza (UTC-3)
df_master['dt_local'] = df_master['dt_utc'] - pd.Timedelta(hours=3)

#Extraindo as informações locais para treinar o modelo
df_master['HORA_LOCAL'] = df_master['dt_local'].dt.hour
df_master['DATA_LOCAL'] = df_master['dt_local'].dt.date

#Descartando as colunas antigas 'DATA', 'HORA' e as auxiliates 'dt_utc', 'HORA_NUM'
df_master = df_master.drop(columns = ['DATA', 'HORA', 'dt_utc', 'HORA_NUM'])


Formalização de linhas com dados vazios

In [70]:
import numpy as np

# Converter o erro do INMET (-9999) em Valor Nulo Real (NaN)
colunas_clima = ['CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD']

df_master[colunas_clima] = df_master[colunas_clima].replace(-9999, np.nan)

# 2. Preenchimento Inteligente: Noite = Radiação 0
# Considerando que em Fortaleza o sol se põe por volta das 18h e nasce às 05:30h
condicao_noite = (df_master['HORA_LOCAL'] >= 18) | (df_master['HORA_LOCAL'] <= 5)

df_master.loc[condicao_noite, 'RAD'] = df_master.loc[condicao_noite, 'RAD'].fillna(0)

# 3. Agora sim, aplica a interpolação linear apenas para buracos pequenos (máx  imo 2h)
df_master['RAD'] = df_master['RAD'].interpolate(method='linear', limit=2)

# 4. Se ainda sobrarem nulos (buracos grandes), preenchemos com 0 para não quebrar o modelo
# (Ou você pode optar por dropna() se quiser apenas dados perfeitos)
df_master['RAD'] = df_master['RAD'].fillna(0)

In [ ]:
import re

def limpar_sujeira_data(valor):
    # Converte para string e remove espaços
    val_str = str(valor).strip()
    
    # REGEX para detectar padrões de data (ex: 04/02/2026 ou 2026-02-04)
    # Se encontrar uma barra '/' ou um hífen '-', é quase certeza que é uma data invasora
    if '/' in val_str or (len(val_str) > 5 and '-' in val_str):
        return np.nan # Transforma a data invasora em nulo
    
    return valor

# Aplicando a limpeza antes de converter para número
cols_clima = ['CHUVA', 'PRESSAO', 'TEMP', 'UMID', 'VENTO', 'RAD']

for col in cols_clima:
    # 1. Remove as datas intrusas
    df_master[col] = df_master[col].apply(limpar_sujeira_data)
    
    # 2. Converte para numérico (coerce transforma o que sobrar de lixo em NaN)
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# 3. Agora os -9999 e as datas viraram NaN. Podemos preencher com a média ou 0
df_master['CHUVA'] = df_master['CHUVA'].fillna(0.0)
df_master['TEMP'] = df_master['TEMP'].fillna(df_master['TEMP'].mean())

if col == 'TEMP' and df_master[col].mean() > 100:
    df_master[col] = df_master[col] / 10

print("Média de Temperatura após remover datas intrusas:", df_master['TEMP'].mean())

Média de Temperatura após remover datas intrusas: 27.333131764245667


In [72]:
print("Média Real de Temperatura esperada (~27):", df_master['TEMP'].mean())

Média Real de Temperatura esperada (~27): 27.333131764245667


Tratamento de Nulos

In [73]:
df_copy = df_master.copy()

# Remover duplicados
df_copy = df_copy.drop_duplicates(subset='dt_local', keep='last')

# Garantir que o DATA_LOCAL É DATATIME
df_copy['DATA_LOCAL'] = pd.to_datetime(df_copy['DATA_LOCAL'])

# Ordenar 
df_copy = df_copy.sort_values('dt_local')

# Colunas
colunas = ['CHUVA','PRESSAO','TEMP','UMID','VENTO','RAD']

# Interpolação
for col in colunas:
    df_copy[col] = df_copy[col].interpolate(limit=6)

# Identificar dias ruins
dias_remover = set()

for col in colunas:
    is_nan = df_copy[col].isna()
    grupos = (is_nan != is_nan.shift()).cumsum()
    
    blocos = df_copy[is_nan].groupby(grupos).size()
    blocos_grandes = blocos[blocos > 6].index
    
    dias = df_copy.loc[grupos.isin(blocos_grandes), 'DATA_LOCAL']
    dias_remover.update(pd.to_datetime(dias).dt.date)

# Remover dias ruins
df_copy = df_copy[~df_copy['DATA_LOCAL'].dt.date.isin(dias_remover)]

# Remover NaN restantes
df_copy = df_copy.dropna()

# Verificar
df_copy.isna().sum()



CHUVA         0
PRESSAO       0
TEMP          0
UMID          0
VENTO         0
RAD           0
dt_local      0
HORA_LOCAL    0
DATA_LOCAL    0
dtype: int64

Criação do Delta

In [74]:
# Ordenar 
df_copy = df_copy.sort_values('dt_local')

# Cria calculo para o DELTA_P e e limita 2 casas decimais
df_copy['DELTA_P'] = (df_copy['PRESSAO'] - df_copy['PRESSAO'].shift(3)).round(2)

# Remover todas as linhas onde a coluna DELTA_P é NaN e reseta o índice
df_copy = df_copy.dropna(subset=['DELTA_P']).reset_index(drop=True)

# Deixar maiúsculo
df_copy = df_copy.rename(columns={'dt_local': 'DT_LOCAL'})

# Mostrar tabela final
df_copy

,CHUVA,PRESSAO,TEMP,UMID,VENTO,RAD,DT_LOCAL,HORA_LOCAL,DATA_LOCAL,DELTA_P
0,0.0,1008.7,27.4,76.0,7.0,0.0,2017-01-01 00:00:00,0,2017-01-01,-0.3
1,0.0,1007.8,27.1,76.0,7.0,0.0,2017-01-01 01:00:00,1,2017-01-01,-1.3
2,0.0,1007.5,27.0,75.0,6.6,0.0,2017-01-01 02:00:00,2,2017-01-01,-1.6
3,0.0,1007.2,26.8,76.0,6.6,0.0,2017-01-01 03:00:00,3,2017-01-01,-1.5
4,0.0,1007.1,26.7,77.0,5.3,0.0,2017-01-01 04:00:00,4,2017-01-01,-0.7
...,...,...,...,...,...,...,...,...,...,...
60619,0.0,1009.7,31.2,60.0,7.0,1.3,2025-01-30 11:00:00,11,2025-01-30,2.6
60620,0.0,1009.3,31.8,55.0,7.3,0.0,2025-01-30 12:00:00,12,2025-01-30,-0.4
60621,0.0,1008.1,31.8,56.0,7.5,0.0,2025-01-30 13:00:00,13,2025-01-30,-1.7
60622,0.0,1006.7,31.3,57.0,7.1,0.0,2025-01-30 14:00:00,14,2025-01-30,-3.0


In [75]:
print(df_copy[['CHUVA', 'TEMP', 'PRESSAO']].describe())

              CHUVA          TEMP       PRESSAO
count  60624.000000  60624.000000  60624.000000
mean       0.178375     27.345623   1009.369968
std        1.343247      2.321055      1.951221
min        0.000000     20.600000   1002.100000
25%        0.000000     25.700000   1008.000000
50%        0.000000     27.000000   1009.300000
75%        0.000000     29.200000   1010.700000
max       50.800000     34.200000   1016.600000


Soma de dados climaticos obtidos por mês 

In [76]:

# Garantir que a coluna de data é do tipo datetime
df_copy['DATA_LOCAL'] = pd.to_datetime(df_copy['DATA_LOCAL'])

# Criar o dicionário de agregação
# Isso permite tratar cada coluna de um jeito diferente no mesmo comando
regras = {
    'CHUVA': 'sum',
    'TEMP': 'mean',
    'UMID': 'mean',
    'PRESSAO': 'mean'
}

# Agrupar por Mês (MS = Month Start) e aplicar as regras
df_mensal = df_copy.resample('MS', on='DATA_LOCAL').agg(regras).reset_index()

#Organizar o nome das colunas para o seu "Dataset Mestre"
df_mensal.columns = ['Ano_mes', 'Chuva_Acumulada', 'Temp_Media_Mensal', 'Umid_Media', 'Pres_Media']

Exportando Dados

In [77]:
#Instala o dataset com os dados formalizados em formato csv
df_mensal.to_csv('dados_formatados/FORTALEZA_DADOS_CLIMATICOS.csv', sep=';', index=False, encoding='utf-8')
print(f"\nSUCESSO! Total de {len(df_mensal)} linhas consolidadas em 'FORTALEZA_DADOS_CLIMATICOS'")


SUCESSO! Total de 97 linhas consolidadas em 'FORTALEZA_DADOS_CLIMATICOS'
